# Food Insecurity Data Pipeline

This notebook cleans, merges, and models food insecurity data across all 3,144 U.S. counties. The pipeline pulls from six raw datasets — BLS unemployment, Census ACS (poverty, disability, homeownership, population), Feeding America meal prices — and produces a single gold-layer output with modeled insecurity rates and annual budget shortfalls.

**Libraries used:** `pandas`, `numpy`, `unicodedata`

---

In [1]:
import pandas as pd

## Unemployment Data (Averaging the unemployment rates)

## Step 1 — Unemployment Data

Source: BLS Local Area Unemployment Statistics (LAUS)

Read the raw Excel file with FIPS codes forced to string type to preserve leading zeros.

In [ ]:
unemployment = pd.read_excel("Raw Data/Unemployment_rate.xlsx", dtype={'State FIPS Code': str, 'County FIPS Code': str}, header=0)

In [3]:
unemployment.head()

,LAUS Code,State FIPS Code,County FIPS Code,County Name/State Abbreviation,Period,Labor Force,Employed,Unemployed,Unemploy-ment Rate (%)
0,CN0100100000000,01,001,"Autauga County, AL",Oct-24,28609,27798,811,2.8
1,CN0100300000000,01,003,"Baldwin County, AL",Oct-24,117918,114611,3307,2.8
2,CN0100500000000,01,005,"Barbour County, AL",Oct-24,8825,8451,374,4.2
3,CN0100700000000,01,007,"Bibb County, AL",Oct-24,8727,8447,280,3.2
4,CN0100900000000,01,009,"Blount County, AL",Oct-24,27134,26401,733,2.7


Drop columns not needed for analysis — LAUS Code, Labor Force, Employed, and raw Unemployed count. We only need the rate.

In [4]:
unemployment = unemployment.drop(["LAUS Code", "Labor Force", "Employed", "Unemployed"], axis=1)

In [5]:
unemployment.tail()

,State FIPS Code,County FIPS Code,County Name/State Abbreviation,Period,Unemploy-ment Rate (%)
45091,72,149,"Villalba Municipio, PR",Nov-25 p,7.9
45092,72,151,"Yabucoa Municipio, PR",Nov-25 p,8.1
45093,72,153,"Yauco Municipio, PR",Nov-25 p,9.1
45094,NaN,NaN,NaN,NaN,"SOURCE: BLS, LAUS"
45095,NaN,NaN,NaN,NaN,"January 16, 2026"


Rows 45094–45095 are footer/summary rows added by BLS. Drop them to keep only county-level records.

In [6]:
unemployment = unemployment.drop([45094, 45095])
unemployment.tail()

,State FIPS Code,County FIPS Code,County Name/State Abbreviation,Period,Unemploy-ment Rate (%)
45089,72,145,"Vega Baja Municipio, PR",Nov-25 p,5.5
45090,72,147,"Vieques Municipio, PR",Nov-25 p,4.9
45091,72,149,"Villalba Municipio, PR",Nov-25 p,7.9
45092,72,151,"Yabucoa Municipio, PR",Nov-25 p,8.1
45093,72,153,"Yauco Municipio, PR",Nov-25 p,9.1


In [7]:
unemployment.dtypes

State FIPS Code                   object
County FIPS Code                  object
County Name/State Abbreviation    object
Period                            object
Unemploy-ment Rate (%)            object
dtype: object

BLS uses an em-dash (`–`) for counties with no October 2025 data. `pd.to_numeric(..., errors='coerce')` converts these to NaN so they can be dropped cleanly.

Also build `full_fips` — a 5-digit FIPS code — by zero-padding the state (2 digits) and county (3 digits) codes. This is the universal geographic identifier used later.

In [8]:
# 1. Force everything to numeric. The '–' for all oct 25 entries will become NaN (null)
unemployment["Unemploy-ment Rate (%)"] = pd.to_numeric(unemployment["Unemploy-ment Rate (%)"], errors='coerce')

# 2. Drop the rows that are now NaN
unemployment = unemployment.dropna(subset=["Unemploy-ment Rate (%)"])

unemployment['full_fips'] = (
    unemployment['State FIPS Code'].astype(str).str.zfill(2) + 
    unemployment['County FIPS Code'].astype(str).str.zfill(3)
)

# 3. Now convert to int
unemployment["Unemploy-ment Rate (%)"] = unemployment["Unemploy-ment Rate (%)"].astype(int)

Some counties appear multiple times across months. Group by FIPS and county name, then take the mean unemployment rate to get a single annual average per county.

In [9]:
unemployment = unemployment.groupby(["full_fips", "County Name/State Abbreviation"]).agg(unemployment_rate = ("Unemploy-ment Rate (%)", "mean")).reset_index()

In [10]:
unemployment.tail()

,full_fips,County Name/State Abbreviation,unemployment_rate
3216,72145,"Vega Baja Municipio, PR",4.857143
3217,72147,"Vieques Municipio, PR",4.357143
3218,72149,"Villalba Municipio, PR",8.285714
3219,72151,"Yabucoa Municipio, PR",8.214286
3220,72153,"Yauco Municipio, PR",9.428571


State abbreviation → full name lookup dictionary. Covers all 50 states plus DC and territories. Used to reconstruct `County, State` strings for joining.

In [11]:
state_map = {
    # 50 States
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas', 'CA': 'California',
    'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware', 'FL': 'Florida', 'GA': 'Georgia',
    'HI': 'Hawaii', 'ID': 'Idaho', 'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa',
    'KS': 'Kansas', 'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland',
    'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi', 'MO': 'Missouri',
    'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada', 'NH': 'New Hampshire', 'NJ': 'New Jersey',
    'NM': 'New Mexico', 'NY': 'New York', 'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio',
    'OK': 'Oklahoma', 'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
    'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah', 'VT': 'Vermont',
    'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia', 'WI': 'Wisconsin', 'WY': 'Wyoming',
    
    # District & Territories
    'DC': 'District of Columbia',
    'PR': 'Puerto Rico',
    'VI': 'Virgin Islands',
    'GU': 'Guam',
    'AS': 'American Samoa',
    'MP': 'Northern Mariana Islands'
}

The BLS column `County Name/State Abbreviation` uses format `County Name, ST`. Split on the last comma, map the abbreviation to the full state name, and reconstruct as `County Name, State Full Name` for consistency across datasets.

In [12]:
# Split the column into two temporary parts
# n=1 ensures we only split at the last comma
split_data = unemployment['County Name/State Abbreviation'].str.split(',', n=1, expand=True)

# 2. Clean up the abbreviation (remove spaces)
# We use the 'state_map' dictionary we created earlier
state_abbr = split_data[1].str.strip().str.upper()
state_full = state_abbr.map(state_map)

# 3. Create the new combined column
# split_data[0] is the County Name
unemployment['County Name/State'] = split_data[0] + ", " + state_full

In [13]:
unemployment.head()

,full_fips,County Name/State Abbreviation,unemployment_rate,County Name/State
0,01001,"Autauga County, AL",2.153846,"Autauga County, Alabama"
1,01003,"Baldwin County, AL",2.307692,"Baldwin County, Alabama"
2,01005,"Barbour County, AL",3.538462,"Barbour County, Alabama"
3,01007,"Bibb County, AL",2.538462,"Bibb County, Alabama"
4,01009,"Blount County, AL",2.230769,"Blount County, Alabama"


In [14]:
unemployment["County Name/State"].isna().sum()

1

In [15]:
missingrows = unemployment[unemployment['County Name/State'].isna()]
print(missingrows)

    full_fips County Name/State Abbreviation  unemployment_rate  \
321     11001           District of Columbia           5.384615   

    County Name/State  
321               NaN  


DC is listed as `District of Columbia` without a comma separator — the split above returns NaN for it. Manually assign the correct value.

In [16]:
# df.loc[condition, column] = value
unemployment.loc[unemployment['County Name/State Abbreviation'] == 'District of Columbia', 'County Name/State'] = 'District of Columbia, District of Columbia'

Drop the original abbreviation column now that we have the full name version. Also convert unemployment rate from integer percentage (e.g. 4) to decimal (0.04) for consistency with other rate columns.

In [17]:
unemployment_final = unemployment.drop("County Name/State Abbreviation", axis=1)

unemployment_final['unemployment_rate'] = unemployment_final['unemployment_rate']/100

In [18]:
unemployment_final.tail()

,full_fips,unemployment_rate,County Name/State
3216,72145,0.048571,"Vega Baja Municipio, Puerto Rico"
3217,72147,0.043571,"Vieques Municipio, Puerto Rico"
3218,72149,0.082857,"Villalba Municipio, Puerto Rico"
3219,72151,0.082143,"Yabucoa Municipio, Puerto Rico"
3220,72153,0.094286,"Yauco Municipio, Puerto Rico"


In [19]:
unemployment_final["County Name/State"].isna().sum()

0

## Poverty Rate (dublicate rows and a few calculated columns)

## Step 2 — Poverty Rate

Source: Census ACS (Table B14006 — non-student adjusted poverty)

This uses Feeding America's refined poverty variable that excludes undergraduate students, as college towns artificially inflate standard poverty rates.

In [20]:
poverty = pd.read_csv("Data/Poverty_rate.csv", header=0)

In [21]:
poverty.head()

,County,Population,Poverty,Poverty_Ugstudents,Unnamed: 4
0,"Autauga County, Alabama",NaN,NaN,NaN,NaN
1,Estimate,"57,409","6,473",134,NaN
2,"Baldwin County, Alabama",NaN,NaN,NaN,NaN
3,Estimate,"237,007","23,381",995,NaN
4,"Barbour County, Alabama",NaN,NaN,NaN,NaN


Drop the unnamed trailing column added by Excel/CSV export.

In [22]:
poverty = poverty.drop("Unnamed: 4", axis=1)

The CSV has staggered rows — county name in one row, estimate in the next. `bfill` (backward fill) pulls the estimate up to the county name row. Then filter out the now-redundant estimate rows.

In [23]:
# bfill(axis=0) pulls data UP from the row below
poverty = poverty.bfill(axis=0, limit=1)

# Filter out the estimate rows
poverty = poverty[~poverty['County'].str.contains('estimate', case=False, na=False)]

In [24]:
poverty = poverty.reset_index()

In [25]:
poverty.tail()

,index,County,Population,Poverty,Poverty_Ugstudents
3217,6434,"Vega Baja Municipio, Puerto Rico","52,600","21,470",841
3218,6436,"Vieques Municipio, Puerto Rico","7,923","4,619",104
3219,6438,"Villalba Municipio, Puerto Rico","20,981","7,972",308
3220,6440,"Yabucoa Municipio, Puerto Rico","28,813","13,615",610
3221,6442,"Yauco Municipio, Puerto Rico","32,129","13,508",444


Convert Population, Poverty, and Poverty_Ugstudents from comma-formatted strings to numeric.

In [26]:
for i in ['Population', 'Poverty', 'Poverty_Ugstudents']:
    poverty[i] = pd.to_numeric(poverty[i].str.replace(',', ''), errors='coerce')

Calculate the non-student poverty rate:
```
poverty_rate = (Total Poverty - Undergraduate Student Poverty) / Total Population
```
This is the exact formula from Feeding America's Map the Meal Gap 2025 methodology.

In [27]:
poverty['poverty_rate'] = (poverty['Poverty'] - poverty['Poverty_Ugstudents'])/poverty['Population']

In [28]:
poverty_final = poverty.drop(["Population", "Poverty", 'Poverty_Ugstudents'], axis = 1)

In [29]:
poverty_final.tail()

,index,County,poverty_rate
3217,6434,"Vega Baja Municipio, Puerto Rico",0.392186
3218,6436,"Vieques Municipio, Puerto Rico",0.569860
3219,6438,"Villalba Municipio, Puerto Rico",0.365283
3220,6440,"Yabucoa Municipio, Puerto Rico",0.451359
3221,6442,"Yauco Municipio, Puerto Rico",0.406611


In [30]:
poverty_final["County"].isna().sum()

0

### Checking for mismatches with county names in poverty and unemployment dfs as county names is the only common column to join them

## Step 3 — Key Standardization

Before merging datasets, check how many county names exist in unemployment but not in poverty (and vice versa). These mismatches will be lost in a join — we need to resolve them.

In [31]:
missing_county =  set(unemployment_final['County Name/State']) - set(poverty['County'])
print(missing_county)

{'Rincon Municipio, Puerto Rico', 'Juneau Borough/city, Alaska', 'Philadelphia County/city, Pennsylvania', 'Rio Grande Municipio, Puerto Rico', 'Guanica Municipio, Puerto Rico', 'San Francisco County/city, California', 'Nantucket County/town, Massachusetts', 'Dona Ana County, New Mexico', 'Manati Municipio, Puerto Rico', 'Loiza Municipio, Puerto Rico', 'Denver County/city, Colorado', 'Mayaguez Municipio, Puerto Rico', 'Broomfield County/city, Colorado', 'Comerio Municipio, Puerto Rico', 'San Sebastian Municipio, Puerto Rico', 'San German Municipio, Puerto Rico', 'Las Marias Municipio, Puerto Rico', 'Canovanas Municipio, Puerto Rico', 'Juana Diaz Municipio, Puerto Rico', 'Anchorage Borough/municipality, Alaska', 'Catano Municipio, Puerto Rico', 'Anasco Municipio, Puerto Rico', 'Sitka Borough/city, Alaska', 'Yakutat Borough/city, Alaska', 'Honolulu County/city, Hawaii', 'Bayamon Municipio, Puerto Rico', 'Penuelas Municipio, Puerto Rico', 'Wrangell Borough/city, Alaska'}


In [32]:
missing_county =  set(poverty['County']) - set(unemployment_final['County Name/State'])
print(missing_county)

{'Nantucket County, Massachusetts', 'Peñuelas Municipio, Puerto Rico', 'Yakutat City and Borough, Alaska', 'Cataño Municipio, Puerto Rico', 'Loíza Municipio, Puerto Rico', 'San Germán Municipio, Puerto Rico', 'Broomfield County, Colorado', 'Canóvanas Municipio, Puerto Rico', 'San Sebastián Municipio, Puerto Rico', 'Manatí Municipio, Puerto Rico', 'Juana Díaz Municipio, Puerto Rico', 'Juneau City and Borough, Alaska', 'Río Grande Municipio, Puerto Rico', 'Mayagüez Municipio, Puerto Rico', 'Doña Ana County, New Mexico', 'Las Marías Municipio, Puerto Rico', 'Wrangell City and Borough, Alaska', 'Anchorage Municipality, Alaska', 'Guánica Municipio, Puerto Rico', 'Denver County, Colorado', 'Kalawao County, Hawaii', 'Añasco Municipio, Puerto Rico', 'Honolulu County, Hawaii', 'Philadelphia County, Pennsylvania', 'San Francisco County, California', 'Sitka City and Borough, Alaska', 'Bayamón Municipio, Puerto Rico', 'Comerío Municipio, Puerto Rico', 'Rincón Municipio, Puerto Rico'}


### The `make_common_key` Function

County names differ across datasets due to suffixes (`County`, `Borough`, `Municipio`), accents, capitalization, and punctuation. This function normalizes all of them to a bare `countyname state` key.

**Key logic:**
- Lowercase and strip punctuation
- Normalize unicode (strips accents)
- Protect 6 Virginia independent cities that share names with counties — detect `city` before stripping it, then re-append after
- Strip noise words: `county`, `borough`, `municipio`, `city`, etc.

In [33]:
import unicodedata

def make_common_key(text):
    if pd.isna(text): return "nan"
    
    # 1. Standardize text
    text = str(text).lower().replace(',', '').replace('/', ' ')
    text = unicodedata.normalize('NFD', text).encode('ascii', 'ignore').decode("utf-8")
    
    # 2. PROTECT THE INDEPENDENT CITIES
    # These 6 cities have same-named counties. We detect "city" before it's stripped.
    cities_to_protect = ['baltimore', 'st. louis', 'fairfax', 'franklin', 'richmond', 'roanoke']
    is_independent_city = any(city in text for city in cities_to_protect) and "city" in text
    
    # 3. Strip "Noise" words
    noise_words = [
        "municipio", "municipality", "city and borough", "borough/city", 
        "county/city", "county/town", "borough", "town", "city", "county"
    ]
    for word in noise_words:
        text = text.replace(word, "")
    
    # Clean extra spaces
    cleaned_text = " ".join(text.split())
    
    # 4. APPEND SUFFIX if it was a protected city
    if is_independent_city:
        return f"{cleaned_text} city"
    
    return cleaned_text

Apply `make_common_key` to both unemployment and poverty DataFrames. This creates a shared `common_key` column used for all subsequent joins.

In [34]:
# Apply to both DataFrames
unemployment_final['common_key'] = unemployment_final['County Name/State'].apply(make_common_key)
poverty_final['common_key'] = poverty_final['County'].apply(make_common_key)

Re-check mismatches after applying the key. Should be zero or near-zero — any remaining mismatches will be dropped in the inner join.

In [35]:
missing_county =  set(unemployment_final['common_key']) - set(poverty_final['common_key'])
print(missing_county)

set()


In [36]:
missing_county =  set(poverty_final['common_key']) - set(unemployment_final['common_key']) 
print(missing_county)

{'kalawao hawaii'}


Kalawao is the smallest county in the U.S. (Hawaii). It is not separately tracked in BLS unemployment — BLS folds it into Maui County.

In [37]:
# Select rows where county is kalawao county, hawaii to check if it exists
filtered_df = poverty_final[poverty_final['County'] == 'Kalawao County, Hawaii']
filtered_df.head()

,index,County,poverty_rate,common_key
550,1100,"Kalawao County, Hawaii",0.227273,kalawao hawaii


Fetch Maui County's unemployment rate to use as the proxy for Kalawao, per standard BLS/Census practice.

In [38]:
# Select rows where county is maui county to check its unemployment rate as kalawao is mixed with that county
filtered_df2 = unemployment_final[unemployment_final['County Name/State'] == 'Maui County, Hawaii']
filtered_df2.head()

,full_fips,unemployment_rate,County Name/State,common_key
551,15009,0.025385,"Maui County, Hawaii",maui hawaii


Manually insert Kalawao County into the unemployment DataFrame using Maui's rate. FIPS 15005 is Kalawao's official code.

In [39]:
# Add the respective row to unemployment_df, the fips code is 15005

# Define the values for Kalawao County
# Using Maui County's rate as the proxy per BLS/Census standard
new_row = {
    'full_fips': '15005',
    'unemployment_rate': 0.02538462,
    'County Name/State': 'Kalawao County, Hawaii',
    'common_key': 'kalawao hawaii'
}

# Insert into the DataFrame
# loc[len(df)] appends it to the very bottom
unemployment_final.loc[len(unemployment_final)] = new_row

In [40]:
unemployment_final.tail()

,full_fips,unemployment_rate,County Name/State,common_key
3217,72147,0.043571,"Vieques Municipio, Puerto Rico",vieques puerto rico
3218,72149,0.082857,"Villalba Municipio, Puerto Rico",villalba puerto rico
3219,72151,0.082143,"Yabucoa Municipio, Puerto Rico",yabucoa puerto rico
3220,72153,0.094286,"Yauco Municipio, Puerto Rico",yauco puerto rico
3221,15005,0.025385,"Kalawao County, Hawaii",kalawao hawaii


Drop the display-friendly county name column — `common_key` is now the join key.

In [41]:
unemployment_final = unemployment_final.drop('County Name/State', axis=1)

Drop the original county name and reset index columns from poverty.

In [42]:
poverty_final = poverty_final.drop(['County', 'index'], axis =1)

## Disability rate

## Step 4 — Disability Rate

Source: Census ACS (Table S1810)

Disability prevalence is a key risk factor in Feeding America's 2020+ model update (coefficient: 0.198).

In [ ]:
disability = pd.read_csv("Raw Data/Disability_rate.csv", header=0)

In [44]:
disability.head(10)

,County,Total civilian noninstitutionalized population
0,"Autauga County, Alabama",NaN
1,Total,NaN
2,Estimate,"58,355"
3,With a disability,NaN
4,Estimate,"9,473"
5,Percent with a disability,NaN
6,Estimate,16.20%
7,"Baldwin County, Alabama",NaN
8,Total,NaN
9,Estimate,"243,859"


Same staggered-row pattern as poverty. Use `bfill` to align rates with county names, then filter out summary and sub-header rows (`estimate`, `total`, `with a disability`).

In [45]:
# bfill(axis=0) pulls data UP from the row below
disability = disability.bfill(axis=0, limit=1)

# Filter out the estimate rows
disability = disability[~disability['County'].str.contains('estimate', case=False, na=False)]
disability = disability[~disability['County'].str.contains('total', case=False, na=False)]
# Keep rows where the County is NOT exactly "with a disability", we did not use "str.contains" because in that case "percent with a disability" also gets deleted
disability = disability[disability['County'].str.strip() != 'With a disability']

In [46]:
disability.head(5)

,County,Total civilian noninstitutionalized population
0,"Autauga County, Alabama",NaN
5,Percent with a disability,16.20%
7,"Baldwin County, Alabama",NaN
12,Percent with a disability,13.20%
14,"Barbour County, Alabama",NaN


Second pass of `bfill` to catch remaining misaligned rows. Remove `Percent with a disability` sub-rows and rename columns to standard names.

In [47]:
# Getting the percentages aligned to the county names
disability = disability.bfill(axis=0, limit=1)

# Removing the redundant rows
disability = disability[disability['County'].str.strip() != 'Percent with a disability']

# renaming column name
disability.columns = ['County', 'disability_rate']

In [48]:
disability.dtypes

County             object
disability_rate    object
dtype: object

In [49]:
disability.tail()

,County,disability_rate
22519,"Vega Baja Municipio, Puerto Rico",21.90%
22526,"Vieques Municipio, Puerto Rico",13.60%
22533,"Villalba Municipio, Puerto Rico",22.90%
22540,"Yabucoa Municipio, Puerto Rico",12.80%
22547,"Yauco Municipio, Puerto Rico",27.60%


Strip the `%` symbol and convert to decimal (divide by 100).

In [50]:
disability['disability_rate'] = pd.to_numeric(disability['disability_rate'].astype(str).str.replace('%', ''), errors='coerce')
disability['disability_rate'] = disability['disability_rate']/100

In [51]:
disability.tail()

,County,disability_rate
22519,"Vega Baja Municipio, Puerto Rico",0.219
22526,"Vieques Municipio, Puerto Rico",0.136
22533,"Villalba Municipio, Puerto Rico",0.229
22540,"Yabucoa Municipio, Puerto Rico",0.128
22547,"Yauco Municipio, Puerto Rico",0.276


In [52]:
disability["County"].isna().sum()

0

Apply `make_common_key` and reset index.

In [53]:
# Use the predefined function to make the county names same for all datasets
disability['common_key'] = disability['County'].apply(make_common_key)

# resetting index
disability_final = disability.reset_index()

In [54]:
disability_final.tail() 

,index,County,disability_rate,common_key
3217,22519,"Vega Baja Municipio, Puerto Rico",0.219,vega baja puerto rico
3218,22526,"Vieques Municipio, Puerto Rico",0.136,vieques puerto rico
3219,22533,"Villalba Municipio, Puerto Rico",0.229,villalba puerto rico
3220,22540,"Yabucoa Municipio, Puerto Rico",0.128,yabucoa puerto rico
3221,22547,"Yauco Municipio, Puerto Rico",0.276,yauco puerto rico


In [55]:
disability_final = disability_final.drop(['County', 'index'], axis = 1)

## Homeownership rate

## Step 5 — Homeownership Rate

Source: Census ACS (Table DP04)

Homeownership is an inverse proxy for economic vulnerability. Coefficient in the model: -0.071.

In [ ]:
homeownership = pd.read_csv("Raw Data/Homeownership_rate.csv", header=0)

In [57]:
homeownership.tail()

,Label (Grouping),HOUSING TENURE!!Occupied housing units!!Owner-occupied
9661,Estimate,"8,597"
9662,Percent,70.7%
9663,"Yauco Municipio, Puerto Rico",NaN
9664,Estimate,"9,653"
9665,Percent,73.0%


Rename columns to standard names.

In [58]:
homeownership.columns = ["County", "homeownership_rate"]

Remove `Estimate` sub-header rows.

In [59]:
homeownership = homeownership[homeownership["County"].str.strip() != "Estimate"]

In [60]:
homeownership.tail()

,County,homeownership_rate
9659,Percent,77.9%
9660,"Yabucoa Municipio, Puerto Rico",NaN
9662,Percent,70.7%
9663,"Yauco Municipio, Puerto Rico",NaN
9665,Percent,73.0%


Second `bfill` pass, remove `Percent` rows, reset index.

In [61]:
homeownership = homeownership.bfill(axis=0, limit=1)

homeownership = homeownership[homeownership["County"].str.strip() != "Percent"]

homeownership_final = homeownership.reset_index()

In [62]:
homeownership_final.tail()

,index,County,homeownership_rate
3217,9651,"Vega Baja Municipio, Puerto Rico",74.8%
3218,9654,"Vieques Municipio, Puerto Rico",66.8%
3219,9657,"Villalba Municipio, Puerto Rico",77.9%
3220,9660,"Yabucoa Municipio, Puerto Rico",70.7%
3221,9663,"Yauco Municipio, Puerto Rico",73.0%


Strip `%` and convert to decimal.

In [63]:
homeownership_final["homeownership_rate"] = homeownership_final["homeownership_rate"].astype(str).str.replace("%", "", regex=False).astype(float)/100

Apply `make_common_key`.

In [64]:
homeownership_final['common_key'] = homeownership_final['County'].apply(make_common_key)

In [65]:
homeownership_final.head()

,index,County,homeownership_rate,common_key
0,0,"Autauga County, Alabama",0.771,autauga alabama
1,3,"Baldwin County, Alabama",0.776,baldwin alabama
2,6,"Barbour County, Alabama",0.682,barbour alabama
3,9,"Bibb County, Alabama",0.792,bibb alabama
4,12,"Blount County, Alabama",0.810,blount alabama


In [66]:
homeownership_final = homeownership_final.drop(['County', 'index'], axis=1)

## Average Meal Prices

## Step 6 — Average Meal Prices

Source: Feeding America / NielsenIQ Cost-of-Food Index

Localized meal costs per county, derived from NielsenIQ in-store scanning data weighted to the USDA Thrifty Food Plan.

In [ ]:
amp = pd.read_csv("Raw Data/average_meal_prices.csv", header=0)

In [68]:
amp.tail()

,"County, State",Cost Per Meal
3139,"Sweetwater County, Wyoming",$3.59
3140,"Teton County, Wyoming",$5.06
3141,"Uinta County, Wyoming",$3.45
3142,"Washakie County, Wyoming",$3.74
3143,"Weston County, Wyoming",$3.72


Strip the `$` symbol and convert to float.

In [69]:
amp['Cost Per Meal ($)'] = amp["Cost Per Meal"].str.replace("$", "", regex=False).astype(float)

In [70]:
amp = amp.drop("Cost Per Meal", axis=1)

Apply `make_common_key`.

In [71]:
amp["common_key"] = amp["County, State"].apply(make_common_key)

In [72]:
amp.tail()

,"County, State",Cost Per Meal ($),common_key
3139,"Sweetwater County, Wyoming",3.59,sweetwater wyoming
3140,"Teton County, Wyoming",5.06,teton wyoming
3141,"Uinta County, Wyoming",3.45,uinta wyoming
3142,"Washakie County, Wyoming",3.74,washakie wyoming
3143,"Weston County, Wyoming",3.72,weston wyoming


Check which counties in poverty are missing from meal prices. Puerto Rico will appear here — it is excluded from Feeding America's meal cost index.

In [73]:
missing_counties = set(poverty_final['common_key']) - set(amp['common_key'])
missing_counties

{'adjuntas puerto rico',
 'aguada puerto rico',
 'aguadilla puerto rico',
 'aguas buenas puerto rico',
 'aibonito puerto rico',
 'anasco puerto rico',
 'arecibo puerto rico',
 'arroyo puerto rico',
 'barceloneta puerto rico',
 'barranquitas puerto rico',
 'bayamon puerto rico',
 'cabo rojo puerto rico',
 'caguas puerto rico',
 'camuy puerto rico',
 'canovanas puerto rico',
 'carolina puerto rico',
 'catano puerto rico',
 'cayey puerto rico',
 'ceiba puerto rico',
 'ciales puerto rico',
 'cidra puerto rico',
 'coamo puerto rico',
 'comerio puerto rico',
 'corozal puerto rico',
 'culebra puerto rico',
 'dorado puerto rico',
 'fajardo puerto rico',
 'florida puerto rico',
 'guanica puerto rico',
 'guayama puerto rico',
 'guayanilla puerto rico',
 'guaynabo puerto rico',
 'gurabo puerto rico',
 'hatillo puerto rico',
 'hormigueros puerto rico',
 'humacao puerto rico',
 'isabela puerto rico',
 'jayuya puerto rico',
 'juana diaz puerto rico',
 'juncos puerto rico',
 'lajas puerto rico',
 'la

Explicitly drop Puerto Rico rows from all DataFrames before the merge. (The inner join would drop them anyway, but this is cleaner and documents intent.)

In [74]:
# Can ignore the below code as when we do inner join the puerto rico rows automatically will not be added to the final merged df

poverty_final = poverty_final[~poverty_final['common_key'].str.contains('puerto rico', case=False, na=False)]
homeownership_final = homeownership_final[~homeownership_final['common_key'].str.contains('puerto rico', case=False, na=False)]
unemployment_final = unemployment_final[~unemployment_final['common_key'].str.contains('puerto rico', case=False, na=False)]
disability_final = disability_final[~disability_final['common_key'].str.contains('puerto rico', case=False, na=False)]

In [75]:
amp_final = amp

In [76]:
print(len(poverty_final))
print(len(homeownership_final))
print(len(unemployment_final))
print(len(disability_final))
print(len(amp_final))

3144
3144
3144
3144
3144


## Population

## Step 7 — Population

Source: Census ACS (Table DP05)

Total county population — used to convert insecurity rate to insecurity count.

In [ ]:
pop = pd.read_csv("Raw Data/population.csv", header=0)

In [78]:
pop.columns = ["County", "population"]

In [79]:
pop.head()

,County,population
0,"Autauga County, Alabama",NaN
1,Estimate,"59,947"
2,Margin of Error,*****
3,"Baldwin County, Alabama",NaN
4,Estimate,"246,989"


Same staggered-row pattern. Remove `Margin of Error` and `Estimate` rows, apply `bfill`, reset index.

In [80]:
pop = pop[pop["County"].str.strip() != "Margin of Error"]

pop = pop.bfill(axis=0, limit=1)

pop = pop[pop["County"].str.strip() != "Estimate"]

pop = pop.reset_index()

Strip commas from formatted numbers and convert to integer.

In [81]:
pop['population'] = pop['population'].str.replace(',', '').astype(int)
pop.head()

,index,County,population
0,0,"Autauga County, Alabama",59947
1,3,"Baldwin County, Alabama",246989
2,6,"Barbour County, Alabama",24643
3,9,"Bibb County, Alabama",22130
4,12,"Blount County, Alabama",59518


Apply `make_common_key` and clean up columns.

In [82]:
pop['common_key'] = pop['County'].apply(make_common_key)
pop_final = pop.drop(["County","index"], axis=1)

pop_final.head()

,population,common_key
0,59947,autauga alabama
1,246989,baldwin alabama
2,24643,barbour alabama
3,22130,bibb alabama
4,59518,blount alabama


## Calculating Food Insecurity Rate

## Step 8 — Master Merge

Inner join all six cleaned DataFrames on `common_key`. Inner join ensures only counties present in ALL datasets are kept — any county missing from even one source is excluded.

In [83]:
merged_df = ( unemployment_final.merge(poverty_final, on='common_key', how='inner')
                                .merge(homeownership_final, on='common_key', how='inner')
                                .merge(disability_final, on='common_key', how='inner')
                                .merge(amp_final, on='common_key', how='inner')
                                .merge(pop_final, on='common_key', how='inner')
            )

In [84]:
merged_df.head()

,full_fips,unemployment_rate,common_key,poverty_rate,homeownership_rate,disability_rate,"County, State",Cost Per Meal ($),population
0,01001,0.021538,autauga alabama,0.110418,0.771,0.162,"Autauga County, Alabama",3.64,59947
1,01003,0.023077,baldwin alabama,0.094453,0.776,0.132,"Baldwin County, Alabama",4.01,246989
2,01005,0.035385,barbour alabama,0.203246,0.682,0.194,"Barbour County, Alabama",3.45,24643
3,01007,0.025385,bibb alabama,0.214390,0.792,0.206,"Bibb County, Alabama",3.36,22130
4,01009,0.022308,blount alabama,0.124275,0.810,0.175,"Blount County, Alabama",3.36,59518


In [85]:
(merged_df == 0).sum()

full_fips             0
unemployment_rate     0
common_key            0
poverty_rate          0
homeownership_rate    1
disability_rate       0
County, State         0
Cost Per Meal ($)     0
population            0
dtype: int64

In [86]:
len(merged_df)

3144

Split the `County, State` column back into separate `county` and `state` columns for dashboard display and regional grouping.

In [87]:
# Split the 'County, State' column into two new columns
# expand=True turns the result into a DataFrame instead of a list
merged_df[['county', 'state']] = merged_df['County, State'].str.split(',', n=1, expand=True)

# 2. Clean up leading/trailing spaces (important for joining!)
merged_df['county'] = merged_df['county'].str.strip()
merged_df['state'] = merged_df['state'].str.strip()

In [88]:
merged_df = merged_df.drop("County, State", axis=1)

Reorder columns into the logical final schema before modeling.

In [89]:
order = ["common_key", "full_fips", "county", "state", "population", "unemployment_rate", "poverty_rate", "disability_rate", "homeownership_rate", "Cost Per Meal ($)"]

merged_df = merged_df[order]

In [90]:
merged_df.head()

,common_key,full_fips,county,state,population,unemployment_rate,poverty_rate,disability_rate,homeownership_rate,Cost Per Meal ($)
0,autauga alabama,01001,Autauga County,Alabama,59947,0.021538,0.110418,0.162,0.771,3.64
1,baldwin alabama,01003,Baldwin County,Alabama,246989,0.023077,0.094453,0.132,0.776,4.01
2,barbour alabama,01005,Barbour County,Alabama,24643,0.035385,0.203246,0.194,0.682,3.45
3,bibb alabama,01007,Bibb County,Alabama,22130,0.025385,0.214390,0.206,0.792,3.36
4,blount alabama,01009,Blount County,Alabama,59518,0.022308,0.124275,0.175,0.810,3.36


## Step 9 — Food Insecurity Rate Model

Implements Feeding America's Map the Meal Gap 2025 multivariate regression formula at county level:

```
FI_rate = 0.101 (constant)
        + 0.013 (2023 year fixed effect)
        + 0.460 × unemployment_rate
        + 0.332 × poverty_rate
        + 0.198 × disability_rate
        - 0.071 × homeownership_rate
```

Coefficients sourced from Appendix Table 1, MMG 2025 Technical Brief (Gundersen et al.).

In [91]:
# The final 2025/2023-based calculation
# Assuming rates are decimals (e.g., 0.12 for 12%)
merged_df['insecurity_rate'] = (
    0.101 + 0.013 + # Constant + 2023 Fixed Effect
    (merged_df['unemployment_rate'] * 0.460) +
    (merged_df['poverty_rate'] * 0.332) +
    (merged_df['disability_rate'] * 0.198) -
    (merged_df['homeownership_rate'] * 0.071)
)

In [92]:
merged_df.tail()

,common_key,full_fips,county,state,population,unemployment_rate,poverty_rate,disability_rate,homeownership_rate,Cost Per Meal ($),insecurity_rate
3139,teton wyoming,56039,Teton County,Wyoming,23396,0.022308,0.052154,0.062,0.583,5.06,0.112460
3140,uinta wyoming,56041,Uinta County,Wyoming,20644,0.032308,0.090623,0.148,0.766,3.45,0.133866
3141,washakie wyoming,56043,Washakie County,Wyoming,7703,0.031538,0.085960,0.141,0.742,3.74,0.132282
3142,weston wyoming,56045,Weston County,Wyoming,6826,0.026154,0.083620,0.144,0.868,3.72,0.120677
3143,kalawao hawaii,15005,Kalawao County,Hawaii,67,0.025385,0.227273,0.258,0.000,5.29,0.252215


In [93]:
distinct_count = merged_df['state'].nunique()

distinct_count

51

State full name → abbreviation lookup. Used to standardize the `state` column to 2-letter codes for the dashboard and region mapping.

In [94]:
state_to_abbr = {
    'alabama': 'AL', 'alaska': 'AK', 'arizona': 'AZ', 'arkansas': 'AR', 'california': 'CA',
    'colorado': 'CO', 'connecticut': 'CT', 'delaware': 'DE', 'district of columbia': 'DC',
    'florida': 'FL', 'georgia': 'GA', 'hawaii': 'HI', 'idaho': 'ID', 'illinois': 'IL',
    'indiana': 'IN', 'iowa': 'IA', 'kansas': 'KS', 'kentucky': 'KY', 'louisiana': 'LA',
    'maine': 'ME', 'maryland': 'MD', 'massachusetts': 'MA', 'michigan': 'MI', 'minnesota': 'MN',
    'mississippi': 'MS', 'missouri': 'MO', 'montana': 'MT', 'nebraska': 'NE', 'nevada': 'NV',
    'new hampshire': 'NH', 'new jersey': 'NJ', 'new mexico': 'NM', 'new york': 'NY',
    'north carolina': 'NC', 'north dakota': 'ND', 'ohio': 'OH', 'oklahoma': 'OK', 'oregon': 'OR',
    'pennsylvania': 'PA', 'rhode island': 'RI', 'south carolina': 'SC', 'south dakota': 'SD',
    'tennessee': 'TN', 'texas': 'TX', 'utah': 'UT', 'vermont': 'VT', 'virginia': 'VA',
    'washington': 'WA', 'west virginia': 'WV', 'wisconsin': 'WI', 'wyoming': 'WY'
}

Map state full names to abbreviations. Flag any that fail to match — these would be lost in the region assignment step.

In [95]:
# Ensure the column is lowercase and stripped to match the dictionary keys
merged_df['state_abbr'] = merged_df['state'].str.lower().str.strip().map(state_to_abbr)

# Check if any states failed to map (returns NaN if not in dictionary)
missing_states = merged_df[merged_df['state_abbr'].isna()]['state'].unique()
if len(missing_states) > 0:
    print(f"Warning: These values did not match the dictionary: {missing_states}")

In [96]:
merged_df['state'] = merged_df['state_abbr']

merged_df = merged_df.drop("state_abbr", axis=1)

Assign each county to one of four Census Bureau regions (NE, MW, S, W) based on its state abbreviation. Region is used later for CPI-U inflation adjustment.

In [97]:
region_map = {
    # Northeast (NE)
    'CT': 'NE', 'ME': 'NE', 'MA': 'NE', 'NH': 'NE', 'RI': 'NE', 'VT': 'NE',
    'NJ': 'NE', 'NY': 'NE', 'PA': 'NE',
    
    # Midwest (MW)
    'IL': 'MW', 'IN': 'MW', 'MI': 'MW', 'OH': 'MW', 'WI': 'MW',
    'IA': 'MW', 'KS': 'MW', 'MN': 'MW', 'MO': 'MW', 'NE': 'MW', 'ND': 'MW', 'SD': 'MW',
    
    # South (S)
    'DE': 'S', 'DC': 'S', 'FL': 'S', 'GA': 'S', 'MD': 'S', 'NC': 'S', 'SC': 'S', 'VA': 'S', 'WV': 'S',
    'AL': 'S', 'KY': 'S', 'MS': 'S', 'TN': 'S',
    'AR': 'S', 'LA': 'S', 'OK': 'S', 'TX': 'S',
    
    # West (W)
    'AZ': 'W', 'CO': 'W', 'ID': 'W', 'MT': 'W', 'NV': 'W', 'NM': 'W', 'UT': 'W', 'WY': 'W',
    'AK': 'W', 'CA': 'W', 'HI': 'W', 'OR': 'W', 'WA': 'W'
}


# Create the 'region' column
merged_df['region'] = merged_df['state'].map(region_map)

# Validation check: Ensure no state was missed
if merged_df['region'].isna().any():
    print("Warning: Some states were not assigned a region!")
    print(merged_df[merged_df['region'].isna()]['state'].unique())

In [98]:
merged_df.head()

,common_key,full_fips,county,state,population,unemployment_rate,poverty_rate,disability_rate,homeownership_rate,Cost Per Meal ($),insecurity_rate,region
0,autauga alabama,01001,Autauga County,AL,59947,0.021538,0.110418,0.162,0.771,3.64,0.137902,S
1,baldwin alabama,01003,Baldwin County,AL,246989,0.023077,0.094453,0.132,0.776,4.01,0.127014,S
2,barbour alabama,01005,Barbour County,AL,24643,0.035385,0.203246,0.194,0.682,3.45,0.187745,S
3,bibb alabama,01007,Bibb County,AL,22130,0.025385,0.214390,0.206,0.792,3.36,0.181410,S
4,blount alabama,01009,Blount County,AL,59518,0.022308,0.124275,0.175,0.810,3.36,0.142661,S


Convert the insecurity rate to a population count.
`np.ceil` rounds up to the nearest whole person — you cannot have a fractional food-insecure individual.

In [99]:
import numpy as np

merged_df['food_insecure_population'] = np.ceil(merged_df['population']*merged_df['insecurity_rate']).astype(int)

In [100]:
merged_df

,common_key,full_fips,county,state,population,unemployment_rate,poverty_rate,disability_rate,homeownership_rate,Cost Per Meal ($),insecurity_rate,region,food_insecure_population
0,autauga alabama,01001,Autauga County,AL,59947,0.021538,0.110418,0.162,0.771,3.64,0.137902,S,8267
1,baldwin alabama,01003,Baldwin County,AL,246989,0.023077,0.094453,0.132,0.776,4.01,0.127014,S,31371
2,barbour alabama,01005,Barbour County,AL,24643,0.035385,0.203246,0.194,0.682,3.45,0.187745,S,4627
3,bibb alabama,01007,Bibb County,AL,22130,0.025385,0.214390,0.206,0.792,3.36,0.181410,S,4015
4,blount alabama,01009,Blount County,AL,59518,0.022308,0.124275,0.175,0.810,3.36,0.142661,S,8491
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3139,teton wyoming,56039,Teton County,WY,23396,0.022308,0.052154,0.062,0.583,5.06,0.112460,W,2632
3140,uinta wyoming,56041,Uinta County,WY,20644,0.032308,0.090623,0.148,0.766,3.45,0.133866,W,2764
3141,washakie wyoming,56043,Washakie County,WY,7703,0.031538,0.085960,0.141,0.742,3.74,0.132282,W,1019
3142,weston wyoming,56045,Weston County,WY,6826,0.026154,0.083620,0.144,0.868,3.72,0.120677,W,824


## Annual Budget Shortfall

## Step 10 — Annual Budget Shortfall

Follows Feeding America's formula exactly:

```
Shortfall = Insecure Population
          × (National Weekly Shortfall × Local Cost Index)
          × 52 weeks
          × (7/12) months
```

The `7/12` factor reflects that food-insecure households experience insecurity an average of 7 months per year (Rabbitt et al., 2024).

The local cost index = `Local Meal Cost / National Average Meal Cost ($3.58)` — this is the CPI-style weight from the NielsenIQ cost-of-food index.

In [101]:
# Define Map the Meal Gap 2023-based constants
NATIONAL_AVG_MEAL_COST = 3.58       # National average meal cost
NATIONAL_AVG_WEEK_SHORTFALL = 22.37 # National average weekly shortfall
WEEKS_IN_YEAR = 52
MONTHS_OF_INSECURITY_FACTOR = 7/12  # Average duration factor per person

# Calculate localized weekly shortfall per person
# Logic: $22.37 * (Local Cost Per Meal / $3.58)
merged_df['weekly_shortfall_per_person'] = (
    NATIONAL_AVG_WEEK_SHORTFALL * (merged_df['Cost Per Meal ($)'] / NATIONAL_AVG_MEAL_COST)
)

# Calculate the total Annual Budget Shortfall
# Logic: Insecure Population * Local Weekly Shortfall * 52 weeks * 7/12 months
merged_df['annual_budget_shortfall'] = (
    merged_df['food_insecure_population'] * merged_df['weekly_shortfall_per_person'] * WEEKS_IN_YEAR * MONTHS_OF_INSECURITY_FACTOR
).round(0).astype(int)

# Drop the intermediate weekly column if not needed
final_df = merged_df.drop(columns=['weekly_shortfall_per_person'])

In [102]:
final_df

,common_key,full_fips,county,state,population,unemployment_rate,poverty_rate,disability_rate,homeownership_rate,Cost Per Meal ($),insecurity_rate,region,food_insecure_population,annual_budget_shortfall
0,autauga alabama,01001,Autauga County,AL,59947,0.021538,0.110418,0.162,0.771,3.64,0.137902,S,8267,5703644
1,baldwin alabama,01003,Baldwin County,AL,246989,0.023077,0.094453,0.132,0.776,4.01,0.127014,S,31371,23843820
2,barbour alabama,01005,Barbour County,AL,24643,0.035385,0.203246,0.194,0.682,3.45,0.187745,S,4627,3025671
3,bibb alabama,01007,Bibb County,AL,22130,0.025385,0.214390,0.206,0.792,3.36,0.181410,S,4015,2556983
4,blount alabama,01009,Blount County,AL,59518,0.022308,0.124275,0.175,0.810,3.36,0.142661,S,8491,5407558
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3139,teton wyoming,56039,Teton County,WY,23396,0.022308,0.052154,0.062,0.583,5.06,0.112460,W,2632,2524291
3140,uinta wyoming,56041,Uinta County,WY,20644,0.032308,0.090623,0.148,0.766,3.45,0.133866,W,2764,1807425
3141,washakie wyoming,56043,Washakie County,WY,7703,0.031538,0.085960,0.141,0.742,3.74,0.132282,W,1019,722352
3142,weston wyoming,56045,Weston County,WY,6826,0.026154,0.083620,0.144,0.868,3.72,0.120677,W,824,580996
